In [1]:
import boto3

s3 = boto3.client("s3", region_name="ap-south-1")

response = s3.list_objects_v2(
    Bucket="maham-crypto-pipeline-2026-246312965173",
    Prefix="silver/trades/"
)

print("AWS connection successful!")
print("Silver files found:", response.get("KeyCount", 0))

AWS connection successful!
Silver files found: 3


In [8]:
import boto3
import pandas as pd
import pyarrow.parquet as pq
import io

s3 = boto3.client("s3", region_name="ap-south-1")

bucket = "maham-crypto-pipeline-2026-246312965173"

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix="silver/trades/"
)

files = [
    obj["Key"]
    for obj in response.get("Contents", [])
    if obj["Key"].endswith(".parquet")
]

print("Parquet files found:", len(files))
print(files)

Parquet files found: 3
['silver/trades/symbol=BTCUSDT/part-00000-20dac357-f60a-44d0-b95e-2093be3019f7.c000.snappy.parquet', 'silver/trades/symbol=ETHUSDT/part-00000-20dac357-f60a-44d0-b95e-2093be3019f7.c000.snappy.parquet', 'silver/trades/symbol=SOLUSDT/part-00000-20dac357-f60a-44d0-b95e-2093be3019f7.c000.snappy.parquet']


In [9]:
dfs = []

for key in files:
    obj = s3.get_object(
        Bucket=bucket,
        Key=key
    )

    table = pq.read_table(
        io.BytesIO(obj["Body"].read())
    )

    temp_df = table.to_pandas()

    # Get symbol from the S3 partition
    symbol = key.split("symbol=")[1].split("/")[0]
    temp_df["symbol"] = symbol

    dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)

print("Total rows:", len(df))
print("Columns:", df.columns.tolist())

df.head()

Total rows: 2166
Columns: ['trade_id', 'price', 'quantity', 'trade_value', 'is_buyer_maker', 'trade_time', 'ingested_at', 'symbol']


,trade_id,price,quantity,trade_value,is_buyer_maker,trade_time,ingested_at,symbol
0,6.733489e+09,84673.78,0.00715,605.417527,True,2026-10-03 06:03:00,2026-10-03 06:04:33,BTCUSDT
1,6.733489e+09,84673.78,0.00006,5.080427,True,2026-10-03 06:03:00,2026-10-03 06:04:34,BTCUSDT
2,6.733489e+09,84673.78,0.00007,5.927165,True,2026-10-03 06:03:00,2026-10-03 06:04:35,BTCUSDT
3,6.733489e+09,84673.78,0.00007,5.927165,True,2026-10-03 06:03:00,2026-10-03 06:04:35,BTCUSDT
4,6.733489e+09,84673.78,0.00007,5.927165,True,2026-10-03 06:03:00,2026-10-03 06:04:36,BTCUSDT


In [10]:
import numpy as np

# Normalize price within each cryptocurrency
df["price_ratio"] = (
    df["price"]
    / df.groupby("symbol")["price"].transform("median")
)

# Log-transform heavily skewed values
df["log_quantity"] = np.log1p(df["quantity"])
df["log_trade_value"] = np.log1p(df["trade_value"])

# Select ML features
features = [
    "price_ratio",
    "log_quantity",
    "log_trade_value"
]

X = df[features].copy()

print("Features prepared:")
print(features)

print("\nMissing values:")
print(X.isnull().sum())

print("\nFeature shape:", X.shape)

Features prepared:
['price_ratio', 'log_quantity', 'log_trade_value']

Missing values:
price_ratio        0
log_quantity       0
log_trade_value    0
dtype: int64

Feature shape: (2166, 3)


In [11]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42
)

model.fit(X)

print("Isolation Forest model trained successfully!")

Isolation Forest model trained successfully!


In [12]:
# Generate anomaly scores
df["anomaly_score"] = model.decision_function(X)

# Generate anomaly labels
# -1 = anomaly
#  1 = normal
df["model_prediction"] = model.predict(X)

# Convert to easy-to-use labels
df["anomaly_label"] = (
    df["model_prediction"] == -1
).astype(int)

print("Anomaly detection completed!")

print("\nAnomaly counts:")
print(df["anomaly_label"].value_counts())

print("\nAnomaly rate:")
print(df["anomaly_label"].mean() * 100, "%")

Anomaly detection completed!

Anomaly counts:
anomaly_label
0    2122
1      44
Name: count, dtype: int64

Anomaly rate:
2.0313942751615883 %


In [13]:
anomalies = df[df["anomaly_label"] == 1].copy()

anomalies = anomalies.sort_values(
    "anomaly_score"
)

print("Total anomalies:", len(anomalies))

anomalies[
    [
        "symbol",
        "price",
        "quantity",
        "trade_value",
        "anomaly_score",
        "trade_time"
    ]
].head(10)

Total anomalies: 44


,symbol,price,quantity,trade_value,anomaly_score,trade_time
2095,ETHUSDT,2697.99,4.6871,12645.748929,-0.132008,2026-10-06 06:52:33
2094,ETHUSDT,2697.98,3.8957,10510.520686,-0.128284,2026-10-06 06:52:27
2155,ETHUSDT,2697.98,3.2298,8713.935804,-0.126693,2026-10-06 06:52:27
1903,ETHUSDT,2698.62,1.4368,3877.377216,-0.120147,2026-10-06 06:46:56
1969,ETHUSDT,2697.98,2.2676,6117.939448,-0.115203,2026-10-06 06:52:27
1581,ETHUSDT,2677.19,1.4927,3996.241513,-0.109972,2026-10-03 05:59:06
1580,ETHUSDT,2677.19,1.3024,3486.772256,-0.105063,2026-10-03 05:59:06
2157,ETHUSDT,2697.99,1.4820,3998.421180,-0.095413,2026-10-06 06:52:33
1902,ETHUSDT,2698.62,1.0000,2698.620000,-0.094674,2026-10-06 06:46:49
2037,ETHUSDT,2697.99,1.4104,3805.245096,-0.092621,2026-10-06 06:52:33


In [15]:
import pyarrow as pa
import pyarrow.parquet as pq
import io

# Convert DataFrame to Parquet
table = pa.Table.from_pandas(df)

buffer = io.BytesIO()

pq.write_table(
    table,
    buffer,
    compression="snappy"
)

# Upload to S3 Gold layer
output_key = "gold/anomalies/crypto_trade_anomalies.parquet"

s3.put_object(
    Bucket=bucket,
    Key=output_key,
    Body=buffer.getvalue()
)

print("ML results saved successfully!")
print(f"s3://{bucket}/{output_key}")

ML results saved successfully!
s3://maham-crypto-pipeline-2026-246312965173/gold/anomalies/crypto_trade_anomalies.parquet
